In [145]:
from feast import FeatureStore
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Any

store = FeatureStore(repo_path=".")

In [146]:
def fetch_historical_features_entity_sql(store: FeatureStore, for_batch_scoring: bool, days: int):
    end_date = (
        datetime.now()
        .replace(microsecond=0, second=0, minute=0)
        .astimezone(tz=timezone.utc)
    )
    start_date = (end_date - timedelta(days=days)).astimezone(tz=timezone.utc)
    # For batch scoring, we want the latest timestamps
    if for_batch_scoring:
        print(
            "Generating a list of all unique entities in a time window for batch scoring"
        )
        # We use a group by since we want all distinct uid.
        entity_sql = f"""
            SELECT
                uid,
                GETDATE() as event_timestamp
            FROM {store.get_data_source("criteo_unprocessed").get_table_query_string()}
            WHERE event_timestamp BETWEEN TIMESTAMP '{start_date.strftime("%Y-%m-%d %H:%M:%S")}' AND TIMESTAMP '{end_date.strftime("%Y-%m-%d %H:%M:%S")}'
            GROUP BY uid
        """
    else:
        print("Generating training data for all entities in a time window")
        # We don't need a group by if we want to generate training data
        entity_sql = f"""
            SELECT
                uid,
                event_timestamp
            FROM {store.get_data_source("criteo_unprocessed").get_table_query_string()}
            WHERE event_timestamp BETWEEN TIMESTAMP '{start_date.strftime("%Y-%m-%d %H:%M:%S")}' AND TIMESTAMP '{end_date.strftime("%Y-%m-%d %H:%M:%S")}'
        """
        

    training_df = store.get_historical_features(
        entity_df=entity_sql,
        features=[
            "impression_unprocessed_view:campaign",
            "impression_unprocessed_view:cat1",
            "impression_unprocessed_view:cat2",
        ],
    ).to_df()
    return training_df

def fetch_online_features(store, source: str = "", entity_rows: List[Dict[str, Any]] = []):
    if source == "":
        features_to_fetch = [
            "impression_unprocessed_view:campaign",
            "impression_unprocessed_view:cat1",
            "impression_unprocessed_view:cat2",
        ]
    else:
        features_to_fetch = store.get_feature_service(source)
    
    online_features = store.get_online_features(
        features=features_to_fetch,
        entity_rows=entity_rows,
    ).to_dict()
    # for key, value in sorted(returned_features.items()):
    #     print(key, " : ", value)
    return online_features

Test batch feature retrieval from the offline store. Performs point in time join, which means that for each entity id at timestamp t, latest values of features are joined within the range of t - ttl

In [148]:
tst = fetch_historical_features_entity_sql(store, False, 2)
tst.head()

INFO:feast.infra.registry.registry:Registry cache expired, so refreshing


Generating training data for all entities in a time window


INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
/Users/paataugrekhelidze/Projects/Recsys/infrastructure/feature_store/Ads/.venv/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()
INFO:feast.infra.registry.registry:Registry cache expired, so refreshing


,uid,event_timestamp,campaign,cat1,cat2
0,14110937,2026-03-19 14:40:01.814668,19602309,138937,9312274
1,7905995,2026-03-19 14:40:08.814668,1428788,30763035,9312274
2,1902072,2026-03-19 14:40:12.814668,73328,28928366,32440047
3,25551042,2026-03-19 14:40:27.814668,28351001,28928366,9312274
4,14908242,2026-03-19 14:40:28.814668,30491418,30763035,26597095


In [119]:
import pandas as pd
entity_df = pd.DataFrame({
    "uid": 19258185,
    # "event_timestamp": [datetime(2026, 3, 20, 12, 0, 0)]  # naive, no tzinfo, also works
    "event_timestamp": [pd.Timestamp(2026, 3, 20, 12, 0, 0).tz_localize("UTC")]  # tzinfo, works
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "impression_unprocessed_view:campaign",
        "impression_unprocessed_view:cat1",
        "impression_unprocessed_view:cat2",
    ],
).to_df()
print(training_df)

INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
/Users/paataugrekhelidze/Projects/Recsys/infrastructure/feature_store/Ads/.venv/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()
INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
/Users/paataugrekhelidze/Projects/Recsys/infrastructure/feature_store/Ads/.venv/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()
INFO:feast.infra.registry

        uid     event_timestamp  campaign    cat1     cat2
0  19258185 2026-03-20 12:00:00  10341182  138937  7477605


Verify that online feature store only pulls the latest values within the date range. The following was tested against "feast materialize 2026-03-19T00:00:00 2026-03-20T00:00:00"

In [ ]:
# identify uid with multiple records in the offline store
tst.groupby("uid").size().reset_index(name="count").sort_values("count", ascending=False)

,uid,count
349476,15827231,78
246877,11171596,60
30753,1402083,54
677245,30610450,54
442585,20042307,50
...,...,...
88714,4009704,1
88715,4009921,1
88716,4009928,1
88717,4010106,1


latest feature values for uid=15827231 in the online store comes from event_timestamp=2026-03-19 22:04:45, not from event_timestamp=2026-03-20 05:32:03, which is correct!

In [153]:
tst[tst["uid"] == 15827231].tail(5)

,uid,event_timestamp,campaign,cat1,cat2
880870,15827231,2026-03-20 05:32:03.814668,2865318,5824233,9312274
907869,15827231,2026-03-19 22:04:45.814668,3960115,30763035,9312274
915008,15827231,2026-03-19 22:05:41.814668,3960115,30763035,9312274
947276,15827231,2026-03-20 11:37:44.814668,14257845,1973606,26597095
961668,15827231,2026-03-20 11:38:20.814668,30368132,5642940,26597095


In [152]:
fetch_online_features(store, source="user_activity", entity_rows=[{"uid": 15827231}])

INFO:feast.infra.registry.registry:Registry cache expired, so refreshing
/Users/paataugrekhelidze/Projects/Recsys/infrastructure/feature_store/Ads/.venv/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()


{'uid': [15827231],
 'cat4': [29196072],
 'cat2': [9312274],
 'cat8': [29196072],
 'cat1': [30763035],
 'cat6': [5824235],
 'cat7': [9312274],
 'campaign': [3960115],
 'cat5': [7477605],
 'cat3': [14402598],
 'cat9': [6755486]}